# WITS Enhanced Inference Pipeline
Inference on Italian WITS dataset with abstraction-aware evaluation metrics and LLM-as-Judge.

**Extension: Semantic Supervision on Italian Wikipedia** - Comparing with original paper on same dataset.

**Metrics:**
- **Abstraction Score**: Measures how much the summary differs from source (1 - n-gram overlap)
- **Compression Ratio**: Generated length / Source length
- **LLM Judge Score**: Llama evaluates quality on 1-5 scale across 4 dimensions

## 1. Setup

In [ ]:
'''# Install deps
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub 'numpy<2.0' 'scipy>=1.10' matplotlib seaborn

!python -m spacy download it_core_news_sm'''

"# Install deps\n!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers     spacy rouge_score bert_score langchain langchain-community langchain-huggingface     huggingface_hub 'numpy<2.0' 'scipy>=1.10' matplotlib seaborn\n\n!python -m spacy download it_core_news_sm"

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import warnings
import logging

# Suppress warning messages
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)
import gc
import re
import torch
import json
import spacy
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

# Load spacy for sentence segmentation
nlp = spacy.load("it_core_news_sm")

## 2. Configuration

In [ ]:
# Login on google colab
from google.colab import userdata
secret = userdata.get('HF_TOKEN')

login(token=secret)

# ArXiv model configuration
SIGEXT_CONFIG = {
    "model_id": "LookUpMark/sigext-wits-it-10k-060t",
    "skip_samples": 25000,
    "threshold": 0.60

}

QUANT_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": torch.float16,
    "bnb_4bit_use_double_quant": True
}
GLOBAL_CONFIG = {
    "llm_model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "judge_model_id": "Qwen/Qwen2.5-14B-Instruct-AWQ",
    "num_test_samples": 100,
    "max_length": 2048,
    "do_sample": False,
    "temperature":0.1,
    "repetition_penalty":1.10,
    "output_dir": "./results_enhanced"
}

GLOBAL_CONFIG["top_p"] = 0.9 if GLOBAL_CONFIG["do_sample"] == True  else 1.0

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)
print(f"Testing with {GLOBAL_CONFIG['num_test_samples']} samples")

Testing with 100 samples


## 3. Enhanced Prompts

In [ ]:
# Italian WITS summarization prompt - Anti-Hallucination Version
SUMMARY_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Sei un riassuntore esperto e FEDELE di articoli enciclopedici. Stai riassumendo informazioni storiche e imparziali.
I tuoi riassunti devono contenere SOLO fatti presenti nella fonte.


REGOLE DI SICUREZZA:
1. Riporta i fatti in modo oggettivo. I fatti riportati NON costituiscono una violazione. L'obiettivo è la sintesi, non la promozione di temi sensibili.
2. Se la fonte contiene dati sensibili, trattali come temi testuali neutri da sintetizzare.
2. Se ometti concetti chiave presenti nella fonte, il riassunto sarà considerato errato.

REGOLE CRITICHE DI FEDELTÀ:
1. Usa SOLO informazioni esplicitamente presenti nel testo fornito.
2. NON aggiungere conoscenze esterne, date, fatti o nomi non presenti nella fonte.
3. Se la fonte non menziona qualcosa, NON devi menzionarlo nemmeno tu.
4. In caso di dubbio su un fatto, OMETTILO piuttosto che inventarlo.
5. NON inferire o estrapolare oltre quanto scritto.

REGOLE DI SCRITTURA:
1. Scrivi UN SOLO paragrafo di MASSIMO 120 parole. NO titoli, NO elenchi.
2. SINTETIZZA e RIFORMULA - mai copiare frasi letteralmente.
3. Inizia con una definizione o contestualizzazione del soggetto.
4. Fondamentale: Concludi sempre l'ultima frase e apponi il punto fermo.
<|eot_id|><|start_header_id|>user<|end_header_id|>

TESTO FONTE:
{source}

CONCETTI CHIAVE DA INTEGRARE (tutti dalla fonte sopra):
{keyphrases}

ISTRUZIONI:
Genera il riassunto seguendo rigorosamente le regole sopra. Assicurati che ogni affermazione sia verificabile nel testo fonte.

<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""


# LLM-as-Judge prompt - G_EVAL SOURCE-AWARE Version (Full Source)
GEVAL_PROMPTS = {
    "faithfulness": """You are an expert summary evaluator.
Task: Evaluate the Faithfulness (1-5) of the Summary based on the Source Text.
GOAL: Determine if the summary contains *only* information supported by the source.

Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Read the summary sentence by sentence.
2. For EACH sentence, search for supporting evidence in the Source. The supporting evidence can be rephrases.
   - IMPORTANT: Look inside headers, lists, and formatted text.
   - IMPORTANT: Accept synonyms (e.g., "capital" matches "capoluogo", "born in" matches dates in brackets).
3. If a claim is not found, verify if it is a logical deduction from the context.
4. If you cannot find a match, explain WHY (e.g. "Masuccio isn't mention anywhere in the source")

Criteria:
- 1: Major Hallucination (Contains specific dates, names, or numbers NOT in the source).
- 3: Mostly faithful, but includes some minor unverifiable details or exaggerations.
- 5: Perfectly Faithful (Every piece of information is supported by the source text, headers, or lists).

Output format:
- Analysis: [Map each summary sentence to a source snippet].
- Verdict: [Explain the score].
- Final: "Score: X" """,

    "completeness": """You are an expert Content Analyst.
Task: Evaluate the Completeness (1-5) of the Summary.
GOAL: Determine if the summary captures the MAIN EVENT/TOPIC of the source, ignoring minor details.

Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Analyze the Source Text and identify the most critical "Who/Where/When/Why".
   - Ignore metadata, file names, or minor side notes.
2. Check if these SPECIFIC core facts are present in the Summary.
   - Allow for rephrasing (e.g., if source says "died in 1990", summary saying "passed away in the 90s" is acceptable coverage).
3. Determine if the summary is completed or leaves the reader confused.

Criteria:
- 1: Irrelevant. Misses the main topic completely.
- 3: Partial. Mentions the topic but misses a crucial fact (e.g., who did it, or the main result).
- 5: Comprehensive. Covers all core entities and main events described in the source.

Output format:
- Key factor identified: [List the Key Facts found in Source].
- Presence in the summary: [YES/NO].
- Final: "Score: X" """,

    "conciseness": """You are an expert Editor.
Task: Evaluate the Conciseness (1-5) of the Summary.
GOAL: Determine if the summary is efficient and dense, or verbose and repetitive.

Input Data:
<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Check for repetitive phrases or redundant adjectives.
2. Check if the sentence structure is unnecessarily complex.
3. Verify if the summary packs a lot of information into few words (High Density).

Criteria:
- 1: Verbose/Repetitive. Uses 20 words where 5 would do. Repeats the same information.
- 3: Average. Readable but contains some filler words or slight redundancy.
- 5: Highly Concise. Every word serves a purpose. High information density.

Output format:
- Analysis: [Brief comment on style].
- Final: "Score: X" """,

    "abstraction": """You are an expert Linguist.
Task: Evaluate the Abstraction (1-5) of the Summary.
GOAL: Determine if the model wrote a new text (Abstractive) or just copied sentences (Extractive).

Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Compare the vocabulary and sentence structure of the Source and Summary.
2. Look for "n-gram overlap". Are there long sequences of words identical to the source?
3. Did the model synthesize information (combine two source sentences into one summary sentence)?

Criteria:
- 1: Copy-Paste. The summary copies long phrases (10+ words) entences taken from the source.
- 3: Mixed. Some rephrasing, but relies heavily on original phrasing/cliches.
- 5: Highly Abstractive. The summary uses entirely new vocabulary and sentence structures to convey the same meaning.

Output format:
- Analysis: [Check for copied phrases]
- Final: "Score: X" """
}

## 4. Helper Functions

In [ ]:
def get_test_data(skip_samples, num_samples):
    print(f"  Loading test data (skipping {skip_samples})...")
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    dataset = dataset.skip(skip_samples)

    test_data = []
    for entry in dataset:
        source = entry['source']
        summary = entry['summary']

        if len(source) < 500 or len(summary) < 50 or len(source) > 10000:
            continue

        test_data.append({"source": source, "reference": summary})

        if len(test_data) >= num_samples:
            break

    print(f"  Test data ready: {len(test_data)} samples")
    return test_data


def load_sigext_model(model_id):
    print(f"  Loading SigExt: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(model_id).to("cuda")
    return model, tokenizer


def load_llm(model_id, quant_config):
    quant_name = "4-bit" if "load_in_4bit" in quant_config else "8-bit"
    print(f"  Loading LLM ({quant_name}): {model_id}...")

    bnb_config = BitsAndBytesConfig(**quant_config)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    tokenizer.add_bos_token = False
    tokenizer.add_eos_token = False

    tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def extract_salient_sentences(text, model, tokenizer, max_length):
    sentences = [sent.text.strip() for sent in nlp(text).sents if len(sent.text.strip()) > 20]

    if not sentences:
        return [], ""

    salient_sentences = []

    for sent in sentences:
        inputs = tokenizer(
            sent,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to("cuda")

        with torch.no_grad():
            logits = model(**inputs).logits

        preds = torch.argmax(logits, dim=2)[0].tolist()

        valid_preds = preds[1:-1] if len(preds) > 2 else preds
        if valid_preds:
            salient_ratio = sum(valid_preds) / len(valid_preds)
            if salient_ratio > 0.75:
                salient_sentences.append(sent)

    keyphrases_text = "\n".join(f"- {s}" for s in salient_sentences)

    return salient_sentences, keyphrases_text


def preprocess_dataset(test_data, sigext_model, sigext_tokenizer, max_length):
    processed_data = []
    for item in tqdm(test_data, desc="    Extracting Salient Sentences"):
        salient_sents, keys_text = extract_salient_sentences(
            item['source'], sigext_model, sigext_tokenizer, max_length
        )
        processed_data.append({
            "source": item['source'],
            "reference": item['reference'],
            "salient_sentences": salient_sents,
            "keyphrases": keys_text
        })
    return processed_data


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

clear_gpu_memory()

## 5. New Abstraction Metrics

In [ ]:
def get_ngrams(text, n=3):
    """Extract n-grams from text."""
    words = text.lower().split()
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]


def compute_abstraction_score(source, generated, n=3):
    """
    Compute abstraction score: 1 - (n-gram overlap with source).
    Higher = more abstractive (less copying).
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)

    if not gen_ngrams:
        return 1.0  # Empty summary = no copying

    copied = sum(1 for ng in gen_ngrams if ng in source_ngrams)
    copy_ratio = copied / len(gen_ngrams)

    return 1.0 - copy_ratio


def compute_compression_ratio(source, generated):
    """Compute compression ratio (lower = more compressed)."""
    if len(source) == 0:
        return 1.0
    return len(generated) / len(source)


def compute_novel_ngrams(source, generated, n=2):
    """
    Compute percentage of n-grams in generated that are NOT in source.
    Higher = more novel content.
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)

    if not gen_ngrams:
        return 0.0

    novel = sum(1 for ng in gen_ngrams if ng not in source_ngrams)
    return novel / len(gen_ngrams)

## 6. LLM-as-Judge Evaluation

In [ ]:
def parse_judge_response(text):
    """Robust JSON parsing for LLM judge responses."""
    import re
    import json

    json_match = re.search(r'Score:\s*(\d)', text, re.IGNORECASE)

    if json_match:
        score = int(match.group(1))
        return max(1, min(5, score))
    return 3


def llm_judge_evaluate(source, generated, judge_chain):
    """Use LLM to judge quality of generated summary against source."""
    scores = {}

    for metric_name, chain in judge_chain.items():
        try:
          inputs = {"generated": generated} # generated for conciseness
          if "{source}" in GEVAL_PROMPTS[metric_name]:
                inputs["source"] = source   # source for faithfulness, completeness and abstraction
          result = chain.invoke(inputs)
          result_text = result.split("assistant<|end_header_id|>")[-1].strip()

          score = parse_judge_response(result_text)
          scores[metric_name] = score

          scores[f"{metric_name}_reason"] = result_text[:200] + "..." # Save 200 characterd

        except Exception as e:
            print(f"      Judge error on {metric_name}: {e}")
            scores[metric_name] = 3 # Default
            scores[f"{metric_name}_reason"] = str(e)

## 7. Enhanced Evaluation Function

In [ ]:
def run_generation_phase(processed_data, summary_chain):
    """Run generation phase."""
    samples = []
    for item in tqdm(processed_data, desc="    Generating Summaries"):
      try:
        res = summary_chain.invoke({"source": item['source'], "keyphrases": item['keyphrases']})
        if "assistant<|end_header_id|>" in res:
          gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()

        samples.append({
            "source": item['source'],
            "reference": item['reference'],
            "salient_sentences": item['salient_sentences'],
            "generated_summary": gen_summary
        })
      except Exception as e:
        print(f"    Error generating summary for item: {e}")
        error_item = item.copy()
        error_item["generated_summary"] = ""
        samples.append(error_item)

    return samples

In [ ]:
def run_enhanced_evaluation(generated_data, judge_chain):
    """Run evaluation with traditional + abstraction + LLM-judge metrics."""

    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

    metrics = {
        # Traditional
        "bert": [], "rouge1": [], "kir": [],
        # Abstraction
        "abstraction": [], "compression": [], "novel_ngrams": [],
        # LLM Judge
        "judge_faithfulness": [], "judge_completeness": [],
        "judge_conciseness": [], "judge_abstraction": []
    }
    samples = []

    for item in tqdm(generated_data, desc="    Evaluating"):
        try:
            gen_summary = item['generated_summary']
            salient_sents = item['salient_sentences']
            source = item['source']
            # === TRADITIONAL METRICS ===

            # ROUGE-1 and ROUGE-L
            rouge_scores = scorer.score(item['reference'], gen_summary)
            metrics["rouge1"].append(rouge_scores['rouge1'].fmeasure)

            # BERTScore (Italian)
            _, _, F1 = bert_score([gen_summary], [item['reference']], lang="it", verbose=False)
            bert_sc = F1.mean().item()
            metrics["bert"].append(bert_sc)

            # KIR
            kir_score = 0.0
            if salient_sents:
                gen_lower = gen_summary.lower()
                hits = 0
                for sent in salient_sents:
                    words = [w.lower() for w in sent.split() if len(w) > 4]
                    if words:
                        word_hits = sum(1 for w in words if w in gen_lower)
                        if word_hits / len(words) > 0.3:
                            hits += 1
                kir_score = hits / len(salient_sents)
            metrics["kir"].append(kir_score)

            # === ABSTRACTION METRICS ===

            abstraction = compute_abstraction_score(item['source'], gen_summary)
            compression = compute_compression_ratio(item['source'], gen_summary)
            novel = compute_novel_ngrams(item['source'], gen_summary)

            metrics["abstraction"].append(abstraction)
            metrics["compression"].append(compression)
            metrics["novel_ngrams"].append(novel)

            # === LLM JUDGE ===

           judge_scores = llm_judge_evaluate(source, gen_summary, judge_chains)

           for key, val in judge_scores.items():
                if key in metrics:
                    metrics[key].append(val)

            # Store sample details
            samples.append({
                "source": item['source'], #[:500] + "..." if len(item['source']) > 500 else item['source'],
                "reference": item['reference'],
                "salient_sentences": salient_sents,
                "generated_summary": gen_summary,
                "scores": {
                    "bert": float(bert_sc),
                    "rouge1": float(rouge_scores['rouge1'].fmeasure),
                                        "kir": float(kir_score),
                    "abstraction": float(abstraction),
                    "compression": float(compression),
                    "novel_ngrams": float(novel),
                    "judge": judge_scores
                }
            })

        except Exception as e:
            print(f"    Error: {e}")
            continue

    return metrics, samples

## 8. Main Evaluation Loop

In [ ]:
print("="*60)
print("PHASE 1: LOADING MODELS")
print("="*60)

# Load SigExt
sigext_model, sigext_tokenizer = load_sigext_model(SIGEXT_CONFIG["model_id"])

# Load test data
test_data = get_test_data(
    SIGEXT_CONFIG["skip_samples"],
    GLOBAL_CONFIG["num_test_samples"]
)

# Extract salient sentences
processed_data = preprocess_dataset(
    test_data,
    sigext_model,
    sigext_tokenizer,
    GLOBAL_CONFIG["max_length"]
)

# Cleanup SigExt
del sigext_model, sigext_tokenizer
clear_gpu_memory()

print("\n" + "="*60)
print("PHASE 2: GENERATION PHASE")
print("="*60)

clear_gpu_memory()

# Load LLM
llm_model, llm_tokenizer = load_llm(
    GLOBAL_CONFIG["llm_model_id"],
    QUANT_CONFIG
)

terminators = [
    llm_tokenizer.eos_token_id,  # <|end_of_text|>
    llm_tokenizer.convert_tokens_to_ids("<|eot_id|>") # <|eot_id|>
]

# Define generation arguments
gen_kwargs = {
    "max_new_tokens": 512,
    "do_sample": GLOBAL_CONFIG["do_sample"],
    "eos_token_id": terminators,
}

if GLOBAL_CONFIG["do_sample"]:
    gen_kwargs.update({
        "temperature": GLOBAL_CONFIG['temperature'],
        "top_p": GLOBAL_CONFIG['top_p'],
        "repetition_penalty": GLOBAL_CONFIG['repetition_penalty'],
    })

# Create text generation pipeline for summary
gen_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    **gen_kwargs
)

llm_summary = HuggingFacePipeline(pipeline=gen_pipe)


# Create chains
summary_prompt = PromptTemplate(
    template=SUMMARY_PROMPT,
    input_variables=["source", "keyphrases"]
)
summary_chain = summary_prompt | llm_summary | StrOutputParser()

data_with_summaries = run_generation_phase(processed_data, summary_chain)

del llm_model, llm_tokenizer, gen_pipe, gen_kwargs, summary_chain
clear_gpu_memory()

print("\n" + "="*60)
print("PHASE 3: EVALUATION PHASE")
print("="*60)

judge_model, judge_tokenizer = load_llm(
    GLOBAL_CONFIG["judge_model_id"],
    QUANT_CONFIG
)

# define the judge argument
judge_kwargs = {
    "max_new_tokens": 512,
    "do_sample": False,
    "eos_token_id": terminators,
}
# Create text generation pipeline for judge
judge_pipe = pipeline(
    "text-generation",
    model=judge_model,
    tokenizer=judge_tokenizer,
    **judge_kwargs
    )

llm_judge = HuggingFacePipeline(pipeline=judge_pipe)

judge_chain = {}

for metric, prompt_txt in GEVAL_PROMPTS.item():
  input_vars =['generated']
  if '{source}' in prompt_txt:
    input_vars.append('source')

  judge_prompt = PromptTemplate(
    template=prompt_txt,
    input_variables=input_vars
  )

  judge_chain[metric] = judge_prompt | llm_judge | StrOutputParser()

metric, samples = run_enhanced_evaluation(processed_data, judge_chain)

del judge_model, judge_tokenizer, judge_pipe, judge_kwargs, llm_judge
clear_gpu_memory()

print("\n" + "="*60)
print("EVALUATION COMPLETE!")
print("="*60)

PHASE 1: LOADING MODELS
  Loading SigExt: LookUpMark/sigext-wits-it-10k-060t...


config.json:   0%|          | 0.00/845 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Error: llm_judge_evaluate() missing 1 required positional argument: 'judge_chain'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Error: llm_judge_evaluate() missing 1 required positional argument: 'judge_chain'

EVALUATION COMPLETE!


## 9. Results Summary

In [ ]:
# Compute and display results
run_info = {
    "timestamp": datetime.now().isoformat(),
    "sigext_model": SIGEXT_CONFIG["model_id"],
    "llm_model": GLOBAL_CONFIG["llm_model_id"],
    "num_samples": len(samples),
    "prompt_type": "optimized_abstractive_v3"
}

if GLOBAL_CONFIG["do_sample"]:
   if "temperature" in GLOBAL_CONFIG:
       run_info["temperature"] = GLOBAL_CONFIG["temperature"]
   if "top_p" in GLOBAL_CONFIG:
       run_info["top_p"] = GLOBAL_CONFIG["top_p"]
   if "repetition_penalty" in GLOBAL_CONFIG:
       run_info["repetition_penalty"] = GLOBAL_CONFIG["repetition_penalty"]
   if 'adapter' in GLOBAL_CONFIG:
       run_info["adapter"] = GLOBAL_CONFIG['adapter']

results = {
    "run_info": run_info,
    "metrics": {}
}

# Aggregate metrics
for m in ["bert", "rouge1", "kir", "abstraction", "compression", "novel_ngrams"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

for m in ["judge_faithfulness", "judge_completeness", "judge_conciseness", "judge_abstraction"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

if metrics.get("judge_faithfulness"):
    overall = [sum(metrics[f"judge_{k}"][i] for k in ["faithfulness", "completeness", "conciseness", "abstraction"])
               for i in range(len(metrics["judge_faithfulness"]))]
    results["metrics"]["judge_overall"] = {"mean": float(np.mean(overall)) / 4, "std": float(np.std(overall)) / 4}

results["samples"] = samples

# Print summary
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

m = results["metrics"]
if "bert" in m:
    print(f"\nTraditional Metrics:")
    print(f"  BERT Score:  {m['bert']['mean']:.4f} +/- {m['bert']['std']:.4f}")
    print(f"  ROUGE-1:     {m['rouge1']['mean']:.4f} +/- {m['rouge1']['std']:.4f}")
    print(f"  KIR:         {m['kir']['mean']:.2%}")

if "abstraction" in m:
    print(f"\nAbstraction Metrics:")
    print(f"  Abstraction: {m['abstraction']['mean']:.4f}")
    print(f"  Novel:       {m['novel_ngrams']['mean']:.2%}")
    print(f"  Compression: {m['compression']['mean']:.2%}")

if "judge_faithfulness" in m:
    print(f"\nLLM Judge (1-5):")
    print(f"  Faithfulness: {m['judge_faithfulness']['mean']:.2f}")
    print(f"  Completeness: {m['judge_completeness']['mean']:.2f}")
    print(f"  Conciseness:  {m['judge_conciseness']['mean']:.2f}")
    print(f"  Abstraction:  {m['judge_abstraction']['mean']:.2f}")
    print(f"  Overall:      {m['judge_overall']['mean']:.2f}")

# Save
output_file = os.path.join(GLOBAL_CONFIG["output_dir"], "results_enhanced.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {output_file}")


RESULTS SUMMARY

Traditional Metrics:
  BERT Score:  0.6616 +/- 0.0379
  ROUGE-1:     0.1995 +/- 0.0951
  KIR:         56.49%

Abstraction Metrics:
  Abstraction: 0.5543
  Novel:       42.11%
  Compression: 31.13%

Saved: ./results_enhanced/results_enhanced.json


## 10. Sample Analysis

In [ ]:
# Sample outputs
print("=" * 60)
print("SAMPLE OUTPUTS")
print("=" * 60)

for idx in [0, len(samples)//2, len(samples)-1]:
    s = samples[idx]
    print(f"\n[Sample {idx}]")
    print(f"Reference: {s['reference'][:200]}...")
    print(f"Generated: {s['generated_summary'][:200]}...")
    sc = s['scores']
    print(f"Scores: BERT={sc['bert']:.2f} ROUGE={sc['rouge1']:.2f} KIR={sc['kir']:.2f} Abstr={sc['abstraction']:.2f}")


SAMPLE OUTPUTS


IndexError: list index out of range

## 11. Cleanup

In [ ]:
# Cleanup
del llm_model, llm_tokenizer, gen_pipe, llm_judge, llm_summary
clear_gpu_memory()

print(" Cleanup complete!")